# Задание

In [178]:
# 1. Выполните сохранение монохромного изображения в виде текстового или бинарного файла.
# 2. Реализуйте алгоритм вейвлет-преобразования Хаара для изображения.
# 3. Выполните квантование высокочастотных компонент (прим., количество квантов  = 4).
# 4. Сохраните получившийся массив значений  в текстовый или бинарный файл в порядке 
#    LL, LH, HL, HH вейвлет-преобразования Хафа. Компоненты LH, HL, HH храните в виде пар (значение, количество повторений).

# Сравните объем памяти, занимаемый исходным изображением (попиксельное хранение), и изображение,
# полученным после преобразования Хафа и сжатием длин серий.

# Решение

In [179]:
import numpy as np
import os
import cv2 as cv

In [180]:
def haar_transform_2d(image):
    img = image.astype(float)
    rows, cols = img.shape

    rows_eff = rows - rows % 2
    cols_eff = cols - cols % 2
    img = img[:rows_eff, :cols_eff]

    for i in range(rows_eff):
        even = img[i, :cols_eff:2]
        odd = img[i, 1:cols_eff:2]
        avg = (even + odd) / 2
        diff = (even - odd) / 2
        img[i, : cols_eff // 2] = avg
        img[i, cols_eff // 2 : cols_eff] = diff

    for j in range(cols_eff):
        even = img[:rows_eff:2, j]
        odd = img[1:rows_eff:2, j]
        avg = (even + odd) / 2
        diff = (even - odd) / 2
        img[: rows_eff // 2, j] = avg
        img[rows_eff // 2 : rows_eff, j] = diff

    return img


def quantize(component, levels=4):
    min_val, max_val = component.min(), component.max()
    if max_val == min_val:
        return np.zeros_like(component, dtype=int)
    bins = np.linspace(min_val, max_val, levels + 1)
    quantized = np.digitize(component, bins) - 1
    quantized = np.clip(quantized, 0, levels - 1)
    return quantized


def rle_encode(arr):
    flat = arr.ravel()
    change_idx = np.where(flat[1:] != flat[:-1])[0] + 1
    splits = np.split(flat, change_idx)
    res = np.array([(s[0], len(s)) for s in splits], dtype=int)
    return res

In [181]:
# Load / Save functions
def load_image_txt(filename):
    return np.loadtxt(filename)


def save_compressed_txt(filename, ll, lh, hl, hh):
    order = [(lh, "LH"), (hl, "HL"), (hh, "HH")]

    with open(filename, "w") as f:
        # LL
        f.write(f"LL {ll.shape[0]} {ll.shape[1]}\n")
        np.savetxt(f, ll, fmt="%d")

        # LH, HL, HH
        for arr, name in order:
            encoded = rle_encode(arr)
            f.write(f"{name} {len(encoded)}\n")
            for val, count in encoded:
                f.write(f"{val} {count}\n")

In [182]:
# 1. Load image
img = cv.imread("monochrome_image.jpg", cv.IMREAD_GRAYSCALE)
np.savetxt("original.txt", img, fmt="%d")

# 2. Perform Haar wavelet transform
transformed = haar_transform_2d(img.copy())
rows, cols = transformed.shape
LL = transformed[: rows // 2, : cols // 2]
LH = transformed[: rows // 2, cols // 2 :]
HL = transformed[rows // 2 :, : cols // 2]
HH = transformed[rows // 2 :, cols // 2 :]

# 3. Quantize and apply RLE
LH_q, HL_q, HH_q = quantize(LH), quantize(HL), quantize(HH)

# 4. Save all components in one text file
save_compressed_txt("haar_compressed.txt", LL, LH_q, HL_q, HH_q)

# 5. Compare file sizes
original_size = os.path.getsize("original.txt")
compressed_size = os.path.getsize("haar_compressed.txt")
print("Original size:", original_size, "bytes")
print("Compressed size:", compressed_size, "bytes")
print(f"Compression ratio: {original_size / compressed_size:.2f}")

Original size: 2389195 bytes
Compressed size: 825781 bytes
Compression ratio: 2.89
